# 附录 L2-06 配套 notebook：模型卡的可核对表

对应课文 [`docs/lessons/appendix/L2-06-模型卡.md`](../docs/lessons/appendix/L2-06-模型卡.md)。

本 notebook **零外部依赖**（只用标准库），把课文里六张模型卡中最容易被读成
"看起来都能用"的四件事，改成可以逐条核对的表：

| 单元 | 问题 |
|---|---|
| 1 | 六张卡到底哪几张**有东西可跑**？（代码 / 权重 / 真值 / 面板四列） |
| 2 | 每张卡的训练条件，**本赛 D/E/F 能不能满足**？（扰动真值那一列） |
| 3 | 四套基因面板**交集是多少**？差集有多大？ |
| 4 | 靶点身份在模型里是**可外推**的还是**查找表**？ |
| 5 | 每张卡能不能用 $\overline{\Delta}/\Gamma$ 四分量口径**归因**？ |

**前置阅读**：[L2-04 简单基线与效应迁移族](../docs/lessons/L2-04-简单基线与效应迁移族.md)（B0–B3 与分量口径）、
[L2-03 基础模型路线](../docs/lessons/L2-03-基础模型路线.md) §6.4（`beta_frac` 分档与 `pert_encoder` 诊断）。

> **边界声明。** 单元 3 的面板是**示意集合**，不是真实面板。真实的 18,533 基因面板
> 需要从 `data/vcc2026-validation/` 读取，Lingshu 的 18,080 与 AlphaCell 的 19,253
> 需要从各自仓库读取。本 notebook **不做那个真实审计**，只演示审计该怎么做。

## 单元 0｜六张卡的数据表

把课文的六张卡压成一张机器可读的表。**这张表是后面所有判定的唯一输入** ——
改这里的字段，后面的结论会跟着变，这是刻意的。

In [ ]:
# 六张卡。字段全部来自课文，证据等级见课文 §4.1。
# "has_code" / "has_weights" 指是否有可获取的官方实现或已发布权重。
# "needs_pert_truth" 指其训练或验证是否需要**目标背景的扰动真值**。
CARDS = [
    dict(id=1, route="分解式", name="perturbation-decomposition",
         paper="P1", has_code=True,  has_weights=False, needs_pert_truth=True,
         panel="2,000 HVG", panel_n=2000, target_id="DepMap 连续向量",
         out_form="伪批量 Δ", licensable=True),
    dict(id=2, route="分解式", name="AdaPert",
         paper="P2", has_code=False, has_weights=False, needs_pert_truth=True,
         panel="3,352/5,000 HVG", panel_n=3352, target_id="STRING 稀疏子图",
         out_form="log1p 表达", licensable=False),
    dict(id=3, route="分解式", name="C3TL",
         paper="P3", has_code=False, has_weights=False, needs_pert_truth=True,
         panel="2,000 HVG", panel_n=2000, target_id="已见扰动聚合",
         out_form="伪批量 Δ", licensable=True),
    dict(id=4, route="生成式", name="X-Cell",
         paper="P4", has_code=False, has_weights=False, needs_pert_truth=True,
         panel="X-Atlas/Pisces 面板", panel_n=0, target_id="多模态 cross-attn",
         out_form="表达谱（未说明）", licensable=True),
    dict(id=5, route="生成式", name="Lingshu-Cell",
         paper="P5", has_code=True,  has_weights=True,  needs_pert_truth=True,
         panel="18,080 genes", panel_n=18080, target_id="基因嵌入 + clue",
         out_form="离散 token", licensable=True),
    dict(id=6, route="世界模型", name="AlphaCell",
         paper="P6", has_code=False, has_weights=False, needs_pert_truth=True,
         panel="19,253 genes", panel_n=19253, target_id="离散 perturbation ID",
         out_form="log1p(CP10K)", licensable=False),
]

print("共 %d 张卡\n" % len(CARDS))
print("%-4s %-8s %-28s %-6s %-6s %-6s" % ("#", "路线", "工作", "代码", "权重", "需真值"))
print("-" * 68)
for c in CARDS:
    print("%-4d %-8s %-28s %-6s %-6s %-6s" % (
        c["id"], c["route"], c["name"],
        "有" if c["has_code"] else "无",
        "有" if c["has_weights"] else "无",
        "是" if c["needs_pert_truth"] else "否",
    ))

## 单元 1｜可执行程度矩阵

课文 §1 说「只有卡 1 与卡 5 有可跑资产」，单元 1 把这个结论**机械算出来**，
而不是靠记忆。四列布尔值分别是：有官方代码、有公开权重、训练条件本赛可满足、
许可允许本项目用途。

**注意最后一列的判据**：只有当「有代码**或**不需要代码（线性/轻量）」时才算可跑 ——
卡 1 的 Ridge/MLP 不需要预训练权重，所以 `has_weights=False` 不扣分。

In [ ]:
# 可跑资产判定：有代码，或者本身不需要权重（线性/轻量模型）。
# 卡 1 的 Ridge/MLP 属于后者。
RUNNABLE = []
for c in CARDS:
    lightweight = c["id"] == 1          # 线性/轻量：无需预训练权重
    runnable = c["has_code"] or lightweight
    RUNNABLE.append(dict(**c, runnable=runnable, lightweight=lightweight))

print("%-28s %-6s %-6s %-8s %-8s" % ("工作", "代码", "权重", "轻量免权重", "可跑"))
print("-" * 64)
for c in RUNNABLE:
    print("%-28s %-6s %-6s %-8s %-8s" % (
        c["name"], "有" if c["has_code"] else "无",
        "有" if c["has_weights"] else "无",
        "是" if c["lightweight"] else "否",
        "✓" if c["runnable"] else "✗",
    ))

n_ok = sum(1 for c in RUNNABLE if c["runnable"])
print("\n可跑资产：%d / %d" % (n_ok, len(CARDS)))
print("可跑的是：%s" % "、".join(c["name"] for c in RUNNABLE if c["runnable"]))
print("\n课文 §1 的结论「只有卡 1 与卡 5 有可跑资产」由上表机械得出。")

## 单元 2｜训练条件判定：本赛 D/E/F 能不能满足

这是六张卡里**真正的分水岭**。课文 §1 末尾说得很直接：六张卡里有四张需要
目标背景的扰动真值，而本赛 D/E/F **一格都没有**。

单元 2 把每张卡的「需要的目标背景扰动比例」摆出来，与本赛的实际值（0%）对照。

In [ ]:
# 目标背景扰动真值的需求比例。数值来自课文各卡「适配缺口」一节：
# 卡 1 / 卡 3 是论文明确写的；卡 4 / 卡 5 是"用同背景已测扰动"这一设计的必然要求；
# 卡 2 接收的是目标背景 NTC + 靶点先验，因此是这里唯一不硬性要求扰动真值的。
NEED_FRAC = {
    1: 0.30,   # [P1] 加目标背景 30% 扰动才改善交互分量
    2: 0.00,   # 接收 NTC + 靶点先验
    3: 0.10,   # [P3] 主实验 8% 训练 + 2% 验证
    4: None,   # [S2] 未给出比例口径，但其零样本宣称需要留出背景
    5: None,   # [P5] H1 用同背景 150 个监督扰动
    6: None,   # [P6] 离散 perturbation ID，无法用于未见靶点
}

THIS_COMP = 0.0   # 本赛 D/E/F：目标背景只给 NTC，扰动真值 = 0

print("本赛 D/E/F 的目标背景扰动真值比例：%.0f%%\n" % (THIS_COMP * 100))
print("%-28s %-14s %-10s %s" % ("工作", "需要比例", "可满足", "理由"))
print("-" * 78)
for c in CARDS:
    frac = NEED_FRAC[c["id"]]
    if frac is None:
        ok, why = "✗", "设计上要求同背景已测扰动 / 已见靶点 ID"
        shown = "未披露"
    elif frac <= THIS_COMP:
        ok, why = "✓", "只依赖 NTC + 靶点先验"
        shown = "%.0f%%" % (frac * 100)
    else:
        ok, why = "✗", "需要目标背景扰动真值，本赛为 0"
        shown = "%.0f%%" % (frac * 100)
    print("%-28s %-14s %-10s %s" % (c["name"], shown, ok, why))

usable = [c["name"] for c in CARDS if NEED_FRAC[c["id"]] is not None and NEED_FRAC[c["id"]] <= THIS_COMP]
print("\n在本赛条件下不需要目标背景扰动真值的：%s" % ("、".join(usable) if usable else "无"))
print("其余全部卡死 —— 不是程度不足，是条件不成立。")

## 单元 3｜面板交集审计（示意）

六张卡里有四套互不相同的基因面板：18,080 / 18,533 / 19,253 / 2,000 HVG。
课文诊断题 2.3 的结论是：**面板差异是词表映射问题，不是「补几个零」**。

单元 3 演示审计该怎么做。**下面用的是小规模示意集合**（基因名用 `G0001` 形式
构造），真实面板必须从 `data/vcc2026-validation/` 与各模型仓库读取。

真实审计需要三步：① 用 Ensembl ID / 符号做**集合运算**，不按列号切；
② 列出交集与**两侧差集**；③ 判定差集基因的填补策略（填零 / 填均值 / 弃用）。

In [ ]:
import random

rng = random.Random(20260918)

# 示意：用一个 18,533 的"比赛面板"，再构造三套与之部分重叠的模型词表面板。
# 重叠率是刻意设成不同的，用来展示差集大小与重叠率不是一回事。
COMP_N = 18533
GENE_UNIVERSE = ["G%05d" % i for i in range(1, 26001)]
comp_panel = set(rng.sample(GENE_UNIVERSE, COMP_N))

def make_panel(overlap_of_comp, n_total):
    """从比赛面板里取一部分，再补上比赛面板之外的基因。"""
    inside = rng.sample(sorted(comp_panel), int(overlap_of_comp * n_total))
    outside = rng.sample(sorted(set(GENE_UNIVERSE) - comp_panel), n_total - len(inside))
    return set(inside) | set(outside)

panels = {
    "Lingshu-Cell (18,080)": make_panel(0.92, 18080),
    "AlphaCell (19,253)":    make_panel(0.88, 19253),
    "AdaPert (3,352 HVG)":   make_panel(0.97, 3352),
}

print("比赛面板：%d 个基因（示意）\n" % len(comp_panel))
print("%-24s %8s %8s %8s %8s" % ("模型面板", "面板大小", "与比赛交集", "模型独有", "比赛缺失"))
print("-" * 66)
for name, p in panels.items():
    inter = p & comp_panel
    only_model = p - comp_panel
    missing = comp_panel - p
    print("%-24s %8d %8d %8d %8d" % (name, len(p), len(inter), len(only_model), len(missing)))

print("\n解读：")
print("  「模型独有」= 模型能预测但比赛面板没有的基因 -> 必须丢弃（会破坏词表/patch 结构）")
print("  「比赛缺失」= 比赛需要但模型给不出的基因     -> 只能填零/填均值/换模型")
print("  两者都无法靠后处理修好：词表长度变化会改变 patch 打包边界。")
print("\n注意：这里的重叠率是示意值，真实数字必须从真实面板算。")

## 单元 4｜靶点身份编码：可外推还是查找表

课文诊断题 2.2 的核心提问方式：**靶点身份在模型里是以什么形式存在的？**
如果是查找表、离散 ID、或依赖靶点自身的表达值，那么「零样本」是名义上的。

单元 4 把六张卡 + B3 的低秩交互项排在一起，逐个判定。**B3 那一行是关键** ——
它是本项目自己的候选，如果 `b_t` 只是 ID embedding，它和 AlphaCell 犯同一个错。

In [ ]:
# 靶点身份的编码形式与"能否外推到未见靶点"。
# 判据：身份参数是否只对训练中见过的靶点定义。
TARGET_ID = [
    ("perturbation-decomposition", "DepMap 连续向量",        True,
     "连续先验，对任意基因都有定义"),
    ("AdaPert",                     "STRING 稀疏子图",       True,
     "外部图按基因查，未见基因也有子图"),
    ("C3TL",                        "已见扰动聚合表示",      False,
     "表示由已测扰动聚合，未见扰动无表示"),
    ("X-Cell",                      "多模态 cross-attn",     True,
     "ESM-2/STRING 等先验对任意基因有定义"),
    ("Lingshu-Cell",                "基因嵌入 + clue",       True,
     "ESM2 嵌入按基因查，未见基因也有向量"),
    ("AlphaCell",                   "离散 perturbation ID",  False,
     "查找表：表里没有的靶点无向量"),
    ("scGPT (L2-03)",               "3 元标记 + 序列位置/表达值", False,
     "靶点表达为零时不可区分"),
    ("B3 低秩交互 (L2-04)",         "b_t 视实现而定",        None,
     "若 b_t 是 ID embedding 则不可外推；若接连续先验则可"),
]

print("%-30s %-26s %-10s %s" % ("模型 / 分量", "靶点身份形式", "可外推", "判据"))
print("-" * 100)
for name, form, ok, why in TARGET_ID:
    tag = "✓" if ok is True else ("✗" if ok is False else "?")
    print("%-30s %-26s %-10s %s" % (name, form, tag, why))

bad = [n for n, _, ok, _ in TARGET_ID if ok is False]
print("\n明确不可外推的：%s" % "、".join(bad))
print("其中 AlphaCell 与 scGPT 是两条独立路线栽在同一处 —— 这是本赛架构设计最该设防的模式。")
print("B3 那一行的 \"?\" 是本项目自己要回答的问题，不是别人的结论。")

## 单元 5｜分量归因能力

最后一张表：每张卡能不能用 [L2-04](../docs/lessons/L2-04-简单基线与效应迁移族.md) 的
四分量口径归因？

$$\Delta_{c,t,g} = \underbrace{\overline{\Delta}_{t,g}}_{\text{共享}} + \underbrace{\Gamma_{c,t,g}}_{\text{交互}} + \varepsilon$$

**分解式路线显式地建模分量，生成式与世界模型把分量埋在分布里。**
这不是优劣判断 —— 它决定了当模型失败时，你**能不能说出它失败在哪一项**。

In [ ]:
# 能否用 (共享分量, 背景特异分量) 归因。
ATTRIB = [
    ("perturbation-decomposition", "分解式",   True,
     "四分量 ANOVA 显式输出 alpha_c / beta_p / gamma_cp"),
    ("AdaPert",                     "分解式",   True,
     "稀疏响应门可解释为'哪些基因响应'，幅度仍分开"),
    ("C3TL",                        "分解式",   True,
     "扰动表示 + 背景表示分离，天然可归因"),
    ("X-Cell",                      "生成式",   False,
     "直接学 P(x|c,t)，分量隐含在生成分布里"),
    ("Lingshu-Cell",                "生成式",   False,
     "掩码离散扩散生成 token，无分量输出"),
    ("AlphaCell",                   "世界模型", False,
     "条件流匹配做状态转移，多一维时间，分量不可分"),
]

print("%-30s %-10s %-8s %s" % ("工作", "路线", "可归因", "理由"))
print("-" * 90)
for name, route, ok, why in ATTRIB:
    print("%-30s %-10s %-8s %s" % (name, route, "✓" if ok else "✗", why))

n_attrib = sum(1 for _, _, ok, _ in ATTRIB if ok)
print("\n可归因：%d / %d，全部来自分解式路线。" % (n_attrib, len(ATTRIB)))
print("含义：生成式路线在本赛上即使分数更高，也无法回答'收益来自共享分量还是交互分量'。")
print("而后者恰恰是 L2-03 §6.4 判断'基础模型赢在哪里'的唯一可证伪口径。")

## 单元 6｜自检清单与三件没做的事

读完本 notebook，你应该能不看课文回答：

1. 六张卡里有几张**有可跑资产**？分别是哪几张？
2. 六张卡里有几张**不需要目标背景扰动真值**？为什么只有那么少？
3. 为什么「面板差 453 个基因」不是补零能解决的？三件事同时发生的是什么？
4. AlphaCell 与 scGPT 的失败模式共同点是什么？B3 的 `b_t` 为什么要提防同一个坑？
5. 生成式路线分数可能更高，但为什么在本项目里仍然难用？缺的是哪一项能力？

In [ ]:
print("=" * 72)
print("自检：本 notebook 的结论与课文对照")
print("=" * 72)
checks = [
    ("有可跑资产", "卡 1（轻量免权重）与卡 5（MIT + 85M 权重）", "课文 §1 末段"),
    ("不需目标背景扰动真值", "只有卡 2；其余全部卡死", "课文 §1 末段 / 单元 2"),
    ("面板差异为何不可补零", "交集/两侧差集 + patch 打包边界同时变", "课文 §2.3 / 单元 3"),
    ("共同的编码陷阱", "身份参数只对已见靶点定义（ID/查找表/依赖观测值）", "课文 §2.2 / 单元 4"),
    ("生成式路线缺的能力", "分量归因（无法说出收益来自 Δ̄ 还是 Γ）", "课文 §0 / 单元 5"),
]
for q, a, ref in checks:
    print("  • %-24s -> %-46s [%s]" % (q, a, ref))

print("\n三件本 notebook 没做的事：")
print("  1. 没有下载任何权重（卡 5 的 85M checkpoint 未拉取）。")
print("  2. 没有做真实面板审计 —— 单元 3 用的是示意集合。")
print("  3. 没有复核任何论文数字（全部为 [P#] 级引用，未复现）。")

print("\n" + "=" * 72)
print("ALL CELLS OK")
print("=" * 72)